In [2]:
from statsbombpy import sb
import pandas as pd
from tqdm import tqdm
import time


In [15]:
comps = sb.competitions()

# Example filter: choose a competition by name
target = comps[(comps["competition_name"] == "Major League Soccer")]

# See seasons available
print(target[["competition_id","season_id","season_name"]].drop_duplicates().head(20))


C:\Users\smart fujitsu\AppData\Roaming\Python\Python312\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


    competition_id  season_id season_name
61              44        107        2023


In [16]:
from statsbombpy import sb
import pandas as pd

competition_id = 44
season_ids = [107]  # List of season IDs for La Liga

all_matches = []

for season in season_ids:
    matches = sb.matches(
        competition_id=competition_id,
        season_id=season
    )
    matches["season_id"] = season   # keep track of season
    all_matches.append(matches)

# Combine all seasons into one DataFrame
matches_df = pd.concat(all_matches, ignore_index=True)

# Extract all match IDs
match_ids = matches_df["match_id"].tolist()

print("Total number of matches:", len(match_ids))

matches_df[["match_id", "home_team", "away_team", "match_date", "season_id"]].head()


C:\Users\smart fujitsu\AppData\Roaming\Python\Python312\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total number of matches: 6


,match_id,home_team,away_team,match_date,season_id
0,3877060,New York Red Bulls,Inter Miami,2023-08-27,107
1,3877090,LAFC,Inter Miami,2023-09-04,107
2,3877194,Charlotte,Inter Miami,2023-10-22,107
3,3877072,Inter Miami,Nashville SC,2023-08-31,107
4,3877170,Inter Miami,Cincinnati,2023-10-08,107


In [ ]:
all_events = []
all_lineups = []

for mid in tqdm(match_ids):
    # EVENTS
    ev = sb.events(match_id=mid)
    ev["match_id"] = mid
    all_events.append(ev)

    # LINEUPS
    lu = sb.lineups(match_id=mid)  # dict: {team_name: dataframe}
    for team_name, df_team in lu.items():
        df_team = df_team.copy()
        df_team["match_id"] = mid
        df_team["team"] = team_name
        all_lineups.append(df_team)

    # small pause to be polite (and avoid hammering)
    time.sleep(0.2)

events_df = pd.concat(all_events, ignore_index=True)
lineups_df = pd.concat(all_lineups, ignore_index=True)

print(events_df.shape, lineups_df.shape)
events_df.head()

  0%|          | 0/6 [00:00<?, ?it/s]C:\Users\smart fujitsu\AppData\Roaming\Python\Python312\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\smart fujitsu\AppData\Roaming\Python\Python312\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
 17%|█▋        | 1/6 [00:01<00:09,  1.98s/it]C:\Users\smart fujitsu\AppData\Roaming\Python\Python312\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\smart fujitsu\AppData\Roaming\Python\Python312\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
 33%|███▎      | 2/6 [00:04<00:08,  2.16s/it]C:\Users\smart fujitsu\AppData\Roaming\Python\Python312\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were n

(21786, 104) (240, 9)


,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,carry_end_location,clearance_aerial_won,clearance_body_part,...,injury_stoppage_in_chain,clearance_other,foul_committed_type,goalkeeper_shot_saved_to_post,shot_saved_to_post,miscontrol_aerial_won,foul_committed_offensive,goalkeeper_punched_out,pass_straight,shot_deflected
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
events_df.to_csv("statsbomb_events_mls.csv", index=False)
matches.to_csv("statsbomb_matches_mls.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np
import ast

# Load data
matches = pd.read_csv("statsbomb_matches_mls.csv")
events  = pd.read_csv("statsbomb_events_mls.csv", low_memory=False)


# Validate required columns
required_matches = {"match_id", "home_team", "away_team", "home_score", "away_score"}
required_events  = {"match_id", "team", "type", "shot_statsbomb_xg"}

missing_m = required_matches - set(matches.columns)
missing_e = required_events  - set(events.columns)

if missing_m:
    raise ValueError(f"Missing columns in matches: {missing_m}")
if missing_e:
    raise ValueError(f"Missing columns in events: {missing_e}")


# Match outcome label
def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    if row["home_score"] < row["away_score"]:
        return "away_win"
    return "draw"

matches["result"] = matches.apply(match_result, axis=1)

# Home/Away mapping (team-level context)
home = matches[["match_id", "home_team"]].rename(columns={"home_team": "team"})
home["home_away"] = "home"

away = matches[["match_id", "away_team"]].rename(columns={"away_team": "team"})
away["home_away"] = "away"

team_side = pd.concat([home, away], ignore_index=True)

# Team x match features from events
event_type = events["type"].astype(str)

events["is_pass"]            = (event_type == "Pass").astype(int)
events["is_shot"]            = (event_type == "Shot").astype(int)
events["is_pressure"]        = (event_type == "Pressure").astype(int)
events["is_foul_committed"]  = (event_type == "Foul Committed").astype(int)
events["is_ball_recovery"]   = (event_type == "Ball Recovery").astype(int)
events["is_duel"]            = (event_type == "Duel").astype(int)
events["is_dribble"]         = (event_type == "Dribble").astype(int)
events["is_interception"]    = (event_type == "Interception").astype(int)
events["is_clearance"]       = (event_type == "Clearance").astype(int)
events["is_tackle"]          = (event_type == "Tackle").astype(int)

# xG (sum over shots; non-shots become 0)
events["shot_statsbomb_xg"] = pd.to_numeric(events["shot_statsbomb_xg"], errors="coerce").fillna(0.0)

# Cards (optional but useful)
red_values = {"Red Card", "Second Yellow", "Second Yellow Card", "Second Yellow (Red)"}

events["foul_committed_card"] = events.get("foul_committed_card", pd.Series([None] * len(events)))
events["bad_behaviour_card"]  = events.get("bad_behaviour_card",  pd.Series([None] * len(events)))

fc = events["foul_committed_card"].astype(str)
bb = events["bad_behaviour_card"].astype(str)

events["is_red_card"]    = ((fc.isin(red_values)) | (bb.isin(red_values))).astype(int)
events["is_yellow_card"] = ((fc == "Yellow Card") | (bb == "Yellow Card")).astype(int)

team_events = (
    events
    .groupby(["match_id", "team"], as_index=False)
    .agg(
        passes=("is_pass", "sum"),
        shots=("is_shot", "sum"),
        xg=("shot_statsbomb_xg", "sum"),
        pressures=("is_pressure", "sum"),
        fouls_committed=("is_foul_committed", "sum"),
        ball_recoveries=("is_ball_recovery", "sum"),
        duels=("is_duel", "sum"),
        dribbles=("is_dribble", "sum"),
        interceptions=("is_interception", "sum"),
        clearances=("is_clearance", "sum"),
        tackles=("is_tackle", "sum"),
        possession_events=("type", "count"),
        red_cards=("is_red_card", "sum"),
        yellow_cards=("is_yellow_card", "sum"),
    )
)

# FORMATION BLOCK (Starting XI + Tactical Shifts)
if "tactics" in events.columns:
    def parse_tactics(x):
        if pd.isna(x):
            return None
        if isinstance(x, dict):
            return x
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except Exception:
                return None
        return None

    events["tactics_parsed"] = events["tactics"].apply(parse_tactics)

    tactics_rows = events[events["tactics_parsed"].notna()].copy()
    tactics_rows["formation"] = tactics_rows["tactics_parsed"].apply(
        lambda d: d.get("formation") if isinstance(d, dict) else None
    )

    # Starting formation per team per match
    starting_formation = (
        tactics_rows[tactics_rows["type"].astype(str) == "Starting XI"]
        .groupby(["match_id", "team"], as_index=False)
        .agg(starting_formation=("formation", "first"))
    )

    # Tactical shifts count + last formation
    tactical_shifts = (
        tactics_rows[tactics_rows["type"].astype(str) == "Tactical Shift"]
        .groupby(["match_id", "team"], as_index=False)
        .agg(
            tactical_shifts=("formation", "count"),
            last_formation=("formation", "last")
        )
    )
else:
    # If tactics column doesn't exist, create empty placeholders
    starting_formation = pd.DataFrame(columns=["match_id", "team", "starting_formation"])
    tactical_shifts = pd.DataFrame(columns=["match_id", "team", "tactical_shifts", "last_formation"])

# Merge features + context + outcomes
final_df = (
    team_events
    .merge(team_side, on=["match_id", "team"], how="left")
    .merge(
        matches[[
            "match_id", "match_date", "competition", "season",
            "home_team", "away_team", "home_score", "away_score", "result"
        ]],
        on="match_id",
        how="left"
    )
    .merge(starting_formation, on=["match_id", "team"], how="left")
    .merge(tactical_shifts, on=["match_id", "team"], how="left")
)

final_df["tactical_shifts"] = final_df["tactical_shifts"].fillna(0).astype(int)

# Team-perspective label (win/draw/loss)
def team_outcome(row):
    if row["result"] == "draw":
        return "draw"
    if row["result"] == "home_win":
        return "win" if row["home_away"] == "home" else "loss"
    return "win" if row["home_away"] == "away" else "loss"

final_df["team_result"] = final_df.apply(team_outcome, axis=1)

# Save
final_df.to_csv("Aggregate_mls.csv", index=False)
print(f"Saved: Aggregate_mls.csv | shape={final_df.shape}")



Saved: Aggregate_mls.csv | shape=(12, 29)
